In [1]:
from pathlib import Path

In [2]:
from typing import List, Tuple
from math import inf
import pandas as pd
import numpy as np

In [4]:
BASE_DIR = Path("..").resolve() 
DATA_DIR = BASE_DIR / "data"
OUT_PATH = DATA_DIR / "food_drug_pairs_silver.csv"

In [5]:
drug_path = DATA_DIR / "drug_clean.csv"
food_subset_path = DATA_DIR / "food_subset.csv"
meal_foods_path = DATA_DIR / "new_foodset.csv"

In [6]:
print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)

BASE_DIR: C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Research\PharmaLink
DATA_DIR: C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Research\PharmaLink\data


In [7]:
drug_clean = pd.read_csv(drug_path)
food_subset = pd.read_csv(food_subset_path)
meal_foods = pd.read_csv(meal_foods_path)

In [8]:
print("drug_clean:", drug_clean.shape)
print("food_subset:", food_subset.shape)
print("meal_foods:", meal_foods.shape)

drug_clean: (192807, 17)
food_subset: (766, 15)
meal_foods: (770, 13)


In [9]:
meal_foods = meal_foods.rename(
    columns={
        "Food_Item": "Food",
        "Calories": "energy",
        "Calories (kcal)": "energy",
        "Protein (g)": "protein",
        "Protein(g)": "protein",
        "Carbohydrate (g)": "carbs",
        "Carbohydrate(g)": "carbs",
        "Fat (g)": "fat",
        "Fat(g)": "fat",
        "Fiber (g)": "fiber",
        "Fiber(g)": "fiber",
        "Meal_Type": "meal_type",
    }
)

In [10]:
required_cols = [
    "Food", "energy", "protein", "fat", "carbs", "fiber",
    "calcium", "iron", "vitamin_c", "vitamin_a",
    "vitamin_k_proxy", "is_alcohol", "is_leafy_green", "meal_type",
]

In [11]:
for col in required_cols:
    if col not in food_subset.columns:
        food_subset[col] = 0
    if col not in meal_foods.columns:
        meal_foods[col] = 0

In [12]:
nutrient_cols = [
    "energy", "protein", "fat", "carbs", "fiber",
    "calcium", "iron", "vitamin_c", "vitamin_a", "vitamin_k_proxy"
]

In [13]:
for col in nutrient_cols:
    food_subset[col] = pd.to_numeric(food_subset[col], errors="coerce").fillna(0.0)
    meal_foods[col] = pd.to_numeric(meal_foods[col], errors="coerce").fillna(0.0)

In [14]:
for flag_col in ["is_alcohol", "is_leafy_green"]:
    food_subset[flag_col] = pd.to_numeric(food_subset[flag_col], errors="coerce").fillna(0).astype(int)
    meal_foods[flag_col] = pd.to_numeric(meal_foods[flag_col], errors="coerce").fillna(0).astype(int)


In [15]:
food_subset = food_subset.fillna(0)
meal_foods = meal_foods.fillna(0)

In [16]:
unified_foods = pd.concat([food_subset, meal_foods], ignore_index=True)
unified_foods["Food"] = unified_foods["Food"].astype(str)

In [17]:
unified_foods = unified_foods.drop_duplicates(subset=["Food"], keep="first").reset_index(drop=True)


In [22]:
print("unified_foods:", unified_foods.shape)
unified_foods.head()

unified_foods: (1468, 22)


,Food,energy,protein,fat,carbs,fiber,calcium,iron,vitamin_c,folate,...,is_alcohol,is_leafy_green,vitamin_k_proxy,meal_type,Category,Carbohydrates (g),Sugars (g),Sodium (mg),Cholesterol (mg),Water_Intake (ml)
0,Beer,43.0,0.0,0.46,3.55,0.0,4.0,0.02,0.0,6.0,...,1,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,Arak,222.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.0,...,1,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,Kasippu,222.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.0,...,1,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,"LIQUOR, TODDY, COCONUT",222.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.0,...,1,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,"LIQUOR, TODDY, KITUL",222.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.0,...,1,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
# Remove unnamed / useless columns
unified_foods = unified_foods.loc[:, ~unified_foods.columns.str.contains("^Unnamed")]


In [23]:
unified_foods.head()


,Food,energy,protein,fat,carbs,fiber,calcium,iron,vitamin_c,folate,...,is_alcohol,is_leafy_green,vitamin_k_proxy,meal_type,Category,Carbohydrates (g),Sugars (g),Sodium (mg),Cholesterol (mg),Water_Intake (ml)
0,Beer,43.0,0.0,0.46,3.55,0.0,4.0,0.02,0.0,6.0,...,1,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,Arak,222.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.0,...,1,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,Kasippu,222.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.0,...,1,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,"LIQUOR, TODDY, COCONUT",222.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.0,...,1,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,"LIQUOR, TODDY, KITUL",222.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.0,...,1,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# Columns we will use for labeling (must not be NaN)
core_cols = [
    "Food", "energy", "protein", "fat", "carbs", "fiber",
    "calcium", "iron", "vitamin_c", "vitamin_a", "vitamin_k_proxy",
    "is_alcohol", "is_leafy_green", "meal_type"
]

# If any core column is missing, create it
for c in core_cols:
    if c not in unified_foods.columns:
        unified_foods[c] = 0

# Fill NaNs only in core columns
unified_foods[core_cols] = unified_foods[core_cols].fillna(0)

# Make sure numeric cols are numeric
num_cols = [
    "energy","protein","fat","carbs","fiber","calcium","iron",
    "vitamin_c","vitamin_a","vitamin_k_proxy"
]
for c in num_cols:
    unified_foods[c] = pd.to_numeric(unified_foods[c], errors="coerce").fillna(0.0)

# Flags as int
for c in ["is_alcohol", "is_leafy_green"]:
    unified_foods[c] = pd.to_numeric(unified_foods[c], errors="coerce").fillna(0).astype(int)

unified_foods["Food"] = unified_foods["Food"].astype(str)

print("unified_foods shape:", unified_foods.shape)
unified_foods[core_cols].head()


unified_foods shape: (1468, 22)


,Food,energy,protein,fat,carbs,fiber,calcium,iron,vitamin_c,vitamin_a,vitamin_k_proxy,is_alcohol,is_leafy_green,meal_type
0,Beer,43.0,0.0,0.46,3.55,0.0,4.0,0.02,0.0,0.0,0,1,0,0
1,Arak,222.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0,1,0,0
2,Kasippu,222.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0,1,0,0
3,"LIQUOR, TODDY, COCONUT",222.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0,1,0,0
4,"LIQUOR, TODDY, KITUL",222.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,0.0,0,1,0,0


In [25]:
# Fix missing columns in drug_clean (if any)
cat_cols = ["Chemical_Class", "Habit_Forming", "Therapeutic_Class", "Action_Class"]
needed_drug_cols = ["Name", "Contains", "combined_text"] + cat_cols

for c in needed_drug_cols:
    if c not in drug_clean.columns:
        drug_clean[c] = ""

# Ensure category columns numeric
for c in cat_cols:
    drug_clean[c] = pd.to_numeric(drug_clean[c], errors="coerce").fillna(0).astype(int)

# Select top N drugs (start with 200)
N_DRUGS = 200
drug_subset = (
    drug_clean.head(N_DRUGS)
    .reset_index()
    .rename(columns={"index": "drug_index"})
)

print("drug_subset shape:", drug_subset.shape)
drug_subset[["drug_index", "Name", "Contains"]].head()


drug_subset shape: (200, 18)


,drug_index,Name,Contains
0,0,Andol 0.5mg Tablet,Haloperidol (0.5mg)
1,1,Avastin 100mg Injection,Bevacizumab (100mg)
2,2,Actorise 40 Injection,Darbepoetin alfa (40mcg)
3,3,Actorise 25 Injection,Darbepoetin alfa (25mcg)
4,4,Actorise 60 Injection,Darbepoetin alfa (60mcg)


In [26]:
cns_keywords = ["antipsychotic", "antidepressant", "antihistamine", "sedative", "antiepileptic"]
abx_keywords = ["amoxycillin", "ciprofloxacin", "tetracycline", "doxycycline"]

def check_food_drug_interaction_with_tags(drug_row, food_row):
    risk = 0
    tags = []

    drug_contains = str(drug_row.get("Contains", "")).lower()
    drug_text = str(drug_row.get("combined_text", "")).lower()

    calcium = float(food_row.get("calcium", 0.0))
    iron = float(food_row.get("iron", 0.0))
    fat = float(food_row.get("fat", 0.0))
    fiber = float(food_row.get("fiber", 0.0))
    vitk = float(food_row.get("vitamin_k_proxy", 0.0))
    is_alcohol = int(food_row.get("is_alcohol", 0))

    if is_alcohol == 1 and any(k in drug_text for k in cns_keywords):
        risk = max(risk, 2)
        tags.append("cns_alcohol")

    if any(k in drug_contains for k in abx_keywords) and calcium > 200:
        risk = max(risk, 1)
        tags.append("calcium_antibiotic")

    if "levothyroxine" in drug_contains and iron > 5:
        risk = max(risk, 2)
        tags.append("iron_levothyroxine")

    if fat > 20 and "empty stomach" in drug_text:
        risk = max(risk, 1)
        tags.append("high_fat_empty_stomach")

    if fiber > 5 and "slow absorption" in drug_text:
        risk = max(risk, 1)
        tags.append("high_fiber_slow_absorption")

    if ("warfarin" in drug_contains or "anticoagulant" in drug_text) and vitk > 100:
        risk = max(risk, 2)
        tags.append("vitk_warfarin")

    return int(risk), ",".join(sorted(set(tags)))


In [27]:
rows = []
total = len(drug_subset) * len(unified_foods)
count = 0

for _, drow in drug_subset.iterrows():
    d_idx = int(drow["drug_index"])

    for _, frow in unified_foods.iterrows():
        severity, tags = check_food_drug_interaction_with_tags(drow, frow)

        rows.append({
            "drug_index": d_idx,
            "drug_name": drow["Name"],
            "drug_contains": drow["Contains"],
            "combined_text": drow.get("combined_text", ""),

            "Chemical_Class": int(drow.get("Chemical_Class", 0)),
            "Habit_Forming": int(drow.get("Habit_Forming", 0)),
            "Therapeutic_Class": int(drow.get("Therapeutic_Class", 0)),
            "Action_Class": int(drow.get("Action_Class", 0)),

            "food_name": frow["Food"],
            "energy": float(frow["energy"]),
            "protein": float(frow["protein"]),
            "fat": float(frow["fat"]),
            "carbs": float(frow["carbs"]),
            "fiber": float(frow["fiber"]),
            "calcium": float(frow["calcium"]),
            "iron": float(frow["iron"]),
            "vitamin_c": float(frow["vitamin_c"]),
            "vitamin_a": float(frow["vitamin_a"]),
            "vitamin_k_proxy": float(frow["vitamin_k_proxy"]),
            "is_alcohol": int(frow["is_alcohol"]),
            "is_leafy_green": int(frow["is_leafy_green"]),
            "meal_type": str(frow["meal_type"]),

            "severity_silver": severity,
            "reason_tags_silver": tags
        })

    count += len(unified_foods)
    print(f"Done drug_index={d_idx}  ({count}/{total})")

silver_df = pd.DataFrame(rows)
print("silver_df shape:", silver_df.shape)

# Save
silver_df.to_csv(OUT_PATH, index=False)
print("Saved file:", OUT_PATH)


Done drug_index=0  (1468/293600)
Done drug_index=1  (2936/293600)
Done drug_index=2  (4404/293600)
Done drug_index=3  (5872/293600)
Done drug_index=4  (7340/293600)
Done drug_index=5  (8808/293600)
Done drug_index=6  (10276/293600)
Done drug_index=7  (11744/293600)
Done drug_index=8  (13212/293600)
Done drug_index=9  (14680/293600)
Done drug_index=10  (16148/293600)
Done drug_index=11  (17616/293600)
Done drug_index=12  (19084/293600)
Done drug_index=13  (20552/293600)
Done drug_index=14  (22020/293600)
Done drug_index=15  (23488/293600)
Done drug_index=16  (24956/293600)
Done drug_index=17  (26424/293600)
Done drug_index=18  (27892/293600)
Done drug_index=19  (29360/293600)
Done drug_index=20  (30828/293600)
Done drug_index=21  (32296/293600)
Done drug_index=22  (33764/293600)
Done drug_index=23  (35232/293600)
Done drug_index=24  (36700/293600)
Done drug_index=25  (38168/293600)
Done drug_index=26  (39636/293600)
Done drug_index=27  (41104/293600)
Done drug_index=28  (42572/293600)
D

In [28]:
silver_df["severity_silver"].value_counts()


severity_silver
0    291448
1      2128
2        24
Name: count, dtype: int64

In [29]:
silver_df["reason_tags_silver"].value_counts().head(15)


reason_tags_silver
                          291448
high_fat_empty_stomach      1582
calcium_antibiotic           546
cns_alcohol                   24
Name: count, dtype: int64

In [30]:
# Separate by class
df_safe = silver_df[silver_df["severity_silver"] == 0]
df_mod  = silver_df[silver_df["severity_silver"] == 1]
df_high = silver_df[silver_df["severity_silver"] == 2]

print(len(df_safe), len(df_mod), len(df_high))


291448 2128 24


In [31]:
gold_safe = df_safe.sample(n=100, random_state=42)
gold_mod  = df_mod.sample(n=100, random_state=42)
gold_high = df_high.copy()  # take all high risk

gold_df = pd.concat([gold_safe, gold_mod, gold_high]).sample(frac=1, random_state=42)

print(gold_df["severity_silver"].value_counts())
gold_df.head()


severity_silver
0    100
1    100
2     24
Name: count, dtype: int64


,drug_index,drug_name,drug_contains,combined_text,Chemical_Class,Habit_Forming,Therapeutic_Class,Action_Class,food_name,energy,...,calcium,iron,vitamin_c,vitamin_a,vitamin_k_proxy,is_alcohol,is_leafy_green,meal_type,severity_silver,reason_tags_silver
88919,60,Avomine Tablet,Promethazine (25mg),Avomine Tablet should be taken with or without...,568,1,15,186,Sweet Potato (1 medium baked),103.0,...,0.0,0.00,0.0,0.0,0.0,0,0,Dinner,0,
221483,150,AB-Flo Capsule,Acebrophylline (100mg),AB-Flo Capsule can be taken with the food in e...,868,1,33,377,Black Pepper (1/2 tsp),3.0,...,0.0,0.00,0.0,0.0,0.0,0,0,Dinner,0,
229916,156,Avanair 100 Tablet,Avanafil (100mg),Avanair 100 Tablet may be taken on an empty st...,710,1,34,0,Pecans (1 oz),196.0,...,0.0,0.00,0.0,0.0,0.0,0,0,Snack,1,high_fat_empty_stomach
229476,156,Avanair 100 Tablet,Avanafil (100mg),Avanair 100 Tablet may be taken on an empty st...,710,1,34,0,"BISCUIT, LEMON PUFF",476.0,...,118.0,2.19,0.0,0.0,0.0,0,0,0,1,high_fat_empty_stomach
180569,123,Amitone 10mg Tablet,Amitriptyline (10mg),Amitone 10mg Tablet is normally taken before b...,273,1,21,385,"LIQUOR, TODDY, PALMYRA",222.0,...,0.0,0.00,0.0,0.0,0.0,1,0,0,2,cns_alcohol


In [32]:
gold_path = DATA_DIR / "food_drug_pairs_gold.xlsx"

gold_df_export = gold_df[[
    "drug_name", "drug_contains", "food_name",
    "severity_silver", "reason_tags_silver"
]].copy()

# Empty columns for manual labels
gold_df_export["severity_gold"] = ""
gold_df_export["reason_tags_gold"] = ""
gold_df_export["reference_source"] = ""

gold_df_export.to_excel(gold_path, index=False)
print("Saved GOLD dataset to:", gold_path)


ModuleNotFoundError: No module named 'openpyxl'

In [33]:
!pip install openpyxl


Defaulting to user installation because normal site-packages is not writeable
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   --------------------

In [35]:
gold_path = DATA_DIR / "food_drug_pairs_gold.csv"

gold_df_export = gold_df[[
    "drug_name", 
    "drug_contains", 
    "food_name",
    "severity_silver", 
    "reason_tags_silver"
]].copy()

# Columns you will manually fill
gold_df_export["severity_gold"] = ""
gold_df_export["reason_tags_gold"] = ""
gold_df_export["reference_source"] = ""

gold_df_export.to_csv(gold_path, index=False)
print("Saved GOLD dataset to:", gold_path)


Saved GOLD dataset to: C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Research\PharmaLink\data\food_drug_pairs_gold.csv


In [36]:
# Downsample SAFE class for training
train_safe = df_safe.sample(n=5000, random_state=42)
train_mod  = df_mod
train_high = df_high

train_df = pd.concat([train_safe, train_mod, train_high]).sample(frac=1, random_state=42)

print(train_df["severity_silver"].value_counts())


severity_silver
0    5000
1    2128
2      24
Name: count, dtype: int64


In [37]:
train_path = DATA_DIR / "food_drug_pairs_train.csv"
train_df.to_csv(train_path, index=False)
print("Saved training dataset:", train_path)


Saved training dataset: C:\Users\User\OneDrive - Sri Lanka Institute of Information Technology\Desktop\Research\PharmaLink\data\food_drug_pairs_train.csv
